# Resources.csv Correction — Projection 2050 — Norte Amazónica

**Scope:** a single, targeted fix -- `DIESEL.avail_exterior` in `Data/2050/{no_transition,
early_access, late_access, early_access_brazil}/C{1-5}/Resources.csv`. Nothing else in
`Resources.csv` is touched. There is no dedicated 2035/2050 resources-generation notebook in this
repo (only `analyse data ramp/sufficiency/resources.ipynb`, for 2025); this notebook is scoped
narrowly to this one correction rather than regenerating the whole file.

**Diagnosis (session 2026-08-05, before any 2050 solve succeeded):** `no_transition_2050` solved
infeasible after the GENSET_DIESEL capacity fix (see `2050/technologies.ipynb`, rule (i)). Two
read-only diagnostics were run: (1) end-use technologies (`REFRIGERATOR_EL`, `LED_BULB`, etc.) were
confirmed still buildable (`f_max` never locked to 0 by rule (a), which only touches electricity
generators) -- not the cause. (2) `DIESEL.avail_exterior` was confirmed to equal *exactly*
2× each cluster's **actual 2025** diesel consumption (not a rounded or hand-set number -- matches
to <0.1% in all 5 clusters). Compared against **no_transition_2035's own actual consumption**
(a fairer reference, since it already reflects one horizon of demand growth), headroom is razor
thin: C4 = 1.001× (essentially zero room for further growth), C1 = 1.10×, C2 = 1.39×, C3 = 1.59×,
C5 = 1.79×. The cap is shared by every diesel end-use (electricity generation, boilers, machinery,
freight), not just `GENSET_DIESEL`, in a 2050 economy larger than 2035's.

**Why this cap has no physical basis to preserve:** diesel is delivered by truck; nothing locally
constrains its supply the way, say, dispersed off-grid capacity is constrained by rollout logistics
(`f_max_prod` on `PV_HS`/`HS_DIESEL`, `2035/technologies.ipynb` rule (h)). `DIESEL.avail_exterior`
exists only to keep the resource-balance row well-conditioned for the solver, not to model a real
ceiling. Freezing it at 2× a **fixed historical (2025) consumption figure**, unrelated to the
horizon being solved, was the actual bug -- not the multiplier itself.

**Fix:** `avail_exterior[c] = 2 × (no_transition_2035's own actual DIESEL consumption in cluster c)`,
read directly from that scenario's solved output
(`case_studies/C1_C2_C3_C4_C5/norte_amazonia_no_transition_2035/outputs/regional_results/
Resources.csv`, column `R_year_exterior`) -- not hand-typed. `no_transition` is deliberately used as
the source for *all four* 2050 scenarios: it is the most diesel-hungry of the three real 2035
runs (no supply-side transition means diesel carries more of the load), so it gives the widest --
least likely to bind -- envelope for every 2050 scenario, including the two where `PV_UTILITY` is
open and diesel draw should be lower anyway.

**Guardrails, per instruction:** every resulting value is checked to be far below `1e14` (the
threshold at which `ESMC_model_AMPL.mod` treats a bound as "inactive"/effectively infinite, per the
same logic already documented for `f_max_prod` in `2035/technologies.ipynb` rule (h)) -- so this is
a real, finite cap, not a disguised "uncapped" placeholder. `1e6` and `1e15` (the two generic
"basically infinite" placeholders used elsewhere in this codebase) are never used here on purpose --
this must stay a physically-traceable number, not a magic constant.


In [1]:
import pandas as pd

ROOT = "../../../EnergyScope_BO_nord_amazonia"
DEPLOY_SCENARIOS_2050 = ["no_transition", "early_access", "late_access", "early_access_brazil"]

# Source: no_transition_2035's own solved output (solve_result_num already verified == 0 for this
# case study in 2050/technologies.ipynb's rule (e) registry construction). R_year_exterior is the
# actual DIESEL throughput consumed across every diesel end-use in that run (GENSET_DIESEL,
# IND_BOILER_DIESEL, *_MACHINERY_DIESEL, freight, etc.), not the input cap -- the real quantity to
# double, per instruction.
DIESEL_SOURCE_PATH = (f"{ROOT}/case_studies/C1_C2_C3_C4_C5/norte_amazonia_no_transition_2035/"
                       f"outputs/regional_results/Resources.csv")

solve_info = pd.read_csv(
    f"{ROOT}/case_studies/C1_C2_C3_C4_C5/norte_amazonia_no_transition_2035/outputs/Solve_info.csv",
    sep=r"\t;\t", header=None, index_col=0, engine="python")
solve_result_num = int(float(solve_info.loc["solve_result_num", 1]))
assert solve_result_num == 0, (
    f"no_transition_2035: solve_result_num={solve_result_num} != 0 -- "
    f"refusing to size DIESEL.avail_exterior from a non-optimal solve")

res_2035 = pd.read_csv(DIESEL_SOURCE_PATH, sep=";")
NEW_DIESEL_AVAIL_EXTERIOR_GWH = {}
for k in range(1, 6):
    row = res_2035[(res_2035["Regions"] == f"C{k}") & (res_2035["Resources"] == "DIESEL")]
    consumption_2035 = float(row["R_year_exterior"].values[0])
    NEW_DIESEL_AVAIL_EXTERIOR_GWH[k] = 2.0 * consumption_2035

print("no_transition_2035 solve_result_num:", solve_result_num)
print()
print(f"{'Cluster':<8}{'no_transition_2035 DIESEL consumption (GWh/y)':>46}{'New avail_exterior = 2x (GWh/y)':>34}")
for k in range(1, 6):
    row = res_2035[(res_2035["Regions"] == f"C{k}") & (res_2035["Resources"] == "DIESEL")]
    consumption_2035 = float(row["R_year_exterior"].values[0])
    print(f"C{k:<7}{consumption_2035:>46.6f}{NEW_DIESEL_AVAIL_EXTERIOR_GWH[k]:>34.6f}")

# Guardrails: every value must be far below 1e14 (the .mod "inactive bound" threshold), and must
# not equal either of the codebase's generic "basically infinite" placeholders (1e6, 1e15) --
# this has to stay a real, traceable number.
for k, v in NEW_DIESEL_AVAIL_EXTERIOR_GWH.items():
    assert v < 1e14, f"C{k}: new avail_exterior={v} is not far below 1e14"
    assert v not in (1e6, 1e15), f"C{k}: new avail_exterior={v} accidentally equals a placeholder constant"
print()
print("ASSERT OK -- all 5 values are far below 1e14 and are not the 1e6/1e15 placeholders")


no_transition_2035 solve_result_num: 0

Cluster  no_transition_2035 DIESEL consumption (GWh/y)   New avail_exterior = 2x (GWh/y)
C1                                            3.243124                          6.486248
C2                                            0.402527                          0.805053
C3                                          358.987745                        717.975489
C4                                          146.174561                        292.349122
C5                                          169.478190                        338.956379

ASSERT OK -- all 5 values are far below 1e14 and are not the 1e6/1e15 placeholders


## Deploy: overwrite `DIESEL.avail_exterior` in `Resources.csv`, all 4 scenarios

Only the `DIESEL` row's `avail_exterior` cell is touched; every other row and column (including
`avail_local`, `gwp_op_local`, `c_op_local`, and C1's `Comment` column, which C2-C5 don't carry) is
read back and re-written unchanged, so the diff is a single value per file.


In [2]:
RESOURCES_PATH_TMPL = f"{ROOT}/Data/2050/{{scenario}}/C{{k}}/Resources.csv"

deployed_diesel_avail = {}  # scenario -> {k: value}
for scenario in DEPLOY_SCENARIOS_2050:
    deployed_diesel_avail[scenario] = {}
    for k in range(1, 6):
        path = RESOURCES_PATH_TMPL.format(scenario=scenario, k=k)
        df = pd.read_csv(path, sep=";", index_col=0)
        df.index = df.index.str.strip()

        before = float(df.loc["DIESEL", "avail_exterior"])
        df.loc["DIESEL", "avail_exterior"] = NEW_DIESEL_AVAIL_EXTERIOR_GWH[k]
        df.to_csv(path, sep=";")

        check = pd.read_csv(path, sep=";", index_col=0)
        check.index = check.index.str.strip()
        after = float(check.loc["DIESEL", "avail_exterior"])
        assert abs(after - NEW_DIESEL_AVAIL_EXTERIOR_GWH[k]) < 1e-9, (
            f"{path}: DIESEL avail_exterior={after}, expected {NEW_DIESEL_AVAIL_EXTERIOR_GWH[k]}")
        deployed_diesel_avail[scenario][k] = after
        print(f"{scenario}/C{k}: DIESEL.avail_exterior {before:.4f} -> {after:.4f} GWh/y")

print()
print("ASSERT OK -- DIESEL.avail_exterior redeployed and verified for all 4 scenarios x C1-C5")


no_transition/C1: DIESEL.avail_exterior 3.5700 -> 6.4862 GWh/y
no_transition/C2: DIESEL.avail_exterior 0.5600 -> 0.8051 GWh/y
no_transition/C3: DIESEL.avail_exterior 570.1500 -> 717.9755 GWh/y
no_transition/C4: DIESEL.avail_exterior 146.2900 -> 292.3491 GWh/y
no_transition/C5: DIESEL.avail_exterior 303.1500 -> 338.9564 GWh/y
early_access/C1: DIESEL.avail_exterior 3.5700 -> 6.4862 GWh/y
early_access/C2: DIESEL.avail_exterior 0.5600 -> 0.8051 GWh/y
early_access/C3: DIESEL.avail_exterior 570.1500 -> 717.9755 GWh/y
early_access/C4: DIESEL.avail_exterior 146.2900 -> 292.3491 GWh/y
early_access/C5: DIESEL.avail_exterior 303.1500 -> 338.9564 GWh/y
late_access/C1: DIESEL.avail_exterior 3.5700 -> 6.4862 GWh/y
late_access/C2: DIESEL.avail_exterior 0.5600 -> 0.8051 GWh/y
late_access/C3: DIESEL.avail_exterior 570.1500 -> 717.9755 GWh/y
late_access/C4: DIESEL.avail_exterior 146.2900 -> 292.3491 GWh/y
late_access/C5: DIESEL.avail_exterior 303.1500 -> 338.9564 GWh/y
early_access_brazil/C1: DIESEL.ava

## Reprint `reg_resources.dat` from the deployed catalog (no solve)

Same mechanism as `2050/technologies.ipynb` Section 12: instantiate `Esmc`, read the just-deployed
`Data/2050/{scenario}/` catalog, apply the same pre-solve preprocessing `scripts/run.py` applies
(`ft_to_drop`, and the `PV_UTILITY`/`BATT_LI` unlock for `early_access`/`late_access`/
`early_access_brazil`), reuse the shared typical-day cache (`algo='read'`, no AMPL call), then
`print_data(indep=True)`. **No `set_esom()`/`solve_esom()` call — no AMPL solve is launched.**


In [3]:
import sys
sys.path.insert(0, r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
from pathlib import Path
from esmc import Esmc

REPRINT_YEAR = 2050
REPRINT_SCENARIO_NAME = {s: s for s in DEPLOY_SCENARIOS_2050}
REPRINT_CASE_STUDY = {s: f"norte_amazonia_{s}_{REPRINT_YEAR}" for s in DEPLOY_SCENARIOS_2050}

FT_TO_DROP = ['BIOMASS_TO_GASOLINE', 'BIOMASS_TO_DIESEL', 'BIOWASTE_TO_GASOLINE', 'BIOWASTE_TO_DIESEL',
              'POWER_TO_GASOLINE', 'POWER_TO_DIESEL', 'H2_TO_GASOLINE', 'H2_TO_DIESEL']
AMPL_PATH = r'C:\Users\valen\AMPL'  # unused by algo='read'

REPRINT_MODELS = {}
for scenario in DEPLOY_SCENARIOS_2050:
    case_study = REPRINT_CASE_STUDY[scenario]
    config = {'case_study': case_study, 'comment': 'DIESEL.avail_exterior reprint (no solve)',
              'regions_names': ['C1', 'C2', 'C3', 'C4', 'C5'],
              'gwp_limit_overall': None, 're_share_primary': None, 'f_perc': True,
              'year': REPRINT_YEAR, 'scenario': REPRINT_SCENARIO_NAME[scenario]}

    my_model = Esmc(config, nbr_td=16)
    current_project = Path(r"C:\Valen\Tfe\EnergyScope_BO_nord_amazonia")
    my_model.project_dir = current_project
    my_model.dat_dir = current_project / 'case_studies' / my_model.space_id / '00_td_dat'
    my_model.cs_dir = current_project / 'case_studies' / my_model.space_id / case_study
    my_model.dat_dir.mkdir(parents=True, exist_ok=True)
    my_model.cs_dir.mkdir(parents=True, exist_ok=True)

    my_model.read_data_indep()
    my_model.init_regions()

    my_model.ref_region.data['Technologies'] = my_model.ref_region.data['Technologies'].drop(index=FT_TO_DROP)
    my_model.data_indep['Layers_in_out'] = my_model.data_indep['Layers_in_out'].drop(index=FT_TO_DROP)
    for r_code, region in my_model.regions.items():
        region.data['Technologies'] = region.data['Technologies'].drop(index=FT_TO_DROP)

    if case_study.startswith('norte_amazonia_early_access_') or case_study.startswith('norte_amazonia_late_access_'):
        for r_code, region in my_model.regions.items():
            region.data['Technologies'].loc['PV_UTILITY', 'f_max'] = 1e15
            region.data['Technologies'].loc['BATT_LI', 'f_max'] = 1e15

    # Pre-print sanity check: in-memory region data (freshly read from the deployed CSVs) must
    # match the just-deployed DIESEL.avail_exterior, before print_data() writes anything.
    for k in range(1, 6):
        region_code = f"C{k}"
        in_memory_diesel = float(my_model.regions[region_code].data['Resources'].loc['DIESEL', 'avail_exterior'])
        expected = deployed_diesel_avail[scenario][k]
        assert abs(in_memory_diesel - expected) < 1e-9, (
            f"{scenario} {region_code}: in-memory DIESEL avail_exterior={in_memory_diesel} after "
            f"init_regions(), expected {expected} (deployed CSV)")

    my_model.init_ta(algo='read', ampl_path=AMPL_PATH)
    my_model.print_td_data()
    my_model.print_data(indep=True)

    REPRINT_MODELS[scenario] = my_model
    print(f"OK -- reg_resources.dat reprinted for {case_study} at {my_model.cs_dir} "
          f"(pre-print in-memory check passed, no solve)")


[INFO    ] (read_data_indep): Read indep data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\00_INDEP


[INFO    ] (init_regions): Initialising regions: C1, C2, C3, C4, C5


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\02_REF_REGION


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\C1


[INFO    ] (read_data): Read data from C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\Data\2050\no_transition\C2


...

[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_td_data): Printing TD data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050\reg_16TD.dat


[INFO    ] (generate_t_h_td): t_h_td and td_count generated


[INFO    ] (print_data): Printing regional data into C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050


OK -- reg_resources.dat reprinted for norte_amazonia_early_access_brazil_2050 at C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050 (pre-print in-memory check passed, no solve)


### Spot-check the reprinted `.dat` text itself

In [4]:
# reg_resources.dat row layout: REGION RESOURCE avail_local avail_exterior gwp_op_local c_op_local
# (same whitespace-split convention already confirmed for reg_technologies.dat) -> avail_exterior
# is field index 3 (0-based).
AVAIL_EXT_FIELD_INDEX = 3

print("=== reg_resources.dat text spot-check (parses the actual avail_exterior value) ===")
for scenario in DEPLOY_SCENARIOS_2050:
    dat_path = REPRINT_MODELS[scenario].cs_dir / "reg_resources.dat"
    lines = dat_path.read_text(encoding="utf-8").splitlines()

    for k in range(1, 6):
        row = next((l for l in lines if l.split() and l.split()[0] == f"C{k}" and l.split()[1] == "DIESEL"), None)
        assert row is not None, f"{scenario} reg_resources.dat: no DIESEL row found for C{k}"
        printed_avail_ext = float(row.split()[AVAIL_EXT_FIELD_INDEX])
        expected = deployed_diesel_avail[scenario][k]
        assert abs(printed_avail_ext - expected) < 1e-6, (
            f"{scenario} reg_resources.dat C{k} DIESEL: printed avail_exterior={printed_avail_ext}, "
            f"expected {expected} (deployed CSV)")
    print(f"OK -- {dat_path}: DIESEL avail_exterior matches the deployed catalog in all 5 clusters")

print()
print("Reprint complete for all 4 scenarios. No set_esom()/solve_esom() call was made -- "
      "no AMPL solve was launched.")


=== reg_resources.dat text spot-check (parses the actual avail_exterior value) ===
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_no_transition_2050\reg_resources.dat: DIESEL avail_exterior matches the deployed catalog in all 5 clusters
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_2050\reg_resources.dat: DIESEL avail_exterior matches the deployed catalog in all 5 clusters
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_late_access_2050\reg_resources.dat: DIESEL avail_exterior matches the deployed catalog in all 5 clusters
OK -- C:\Valen\Tfe\EnergyScope_BO_nord_amazonia\case_studies\C1_C2_C3_C4_C5\norte_amazonia_early_access_brazil_2050\reg_resources.dat: DIESEL avail_exterior matches the deployed catalog in all 5 clusters

Reprint complete for all 4 scenarios. No set_esom()/solve_esom() call was made -- no AMPL solve was launched.
